# Codage, stockage et calculs sur un ordinateur

## Codage de l'information

### Compter en binaire

Un code binaire représente les informations en utilisant uniquement 2 symboles : `0` et `1`. Sur 8 bits (=1 octet), on peut représenter un nombre compris entre 0 et 255. 
En chaque position `m`, on représente la puissance `m-1` de la base. En base 10, qui est la base que l'on utilise au quotidien, on a 
```
10^0 = 1
10^1 = 10
...
10^5 = 100000.
```
En base 2, on ne considère pas des puissances de 10 mais des puissances de 2 : 
```
2^0 = 1
2^1 = 2
...
2^5 = 32.
```
Donc, si l'on a 8 bit, on peut aller jusqu'à 255, avec les premiers nombres donc codés :
```
0 = 0^0
1 = 2^0
10 = 2^1 + 0^0 = 2
11 =  2^1 + 2^0 = 3
100 = 2^2 + 0^1 + 0^0 = 4
101 = 2^2 + 0^1 + 2^0 = 5
110 = 2^2 + 2^1 + 2^0 = 6
...
101011 = 2^5 + 0^4 + 2^3 + 0^2 + 2^1 + 2^0 = 43
...
11111111 = 2^7 + 2^6 + ... + 2^1 + 2^0 = 255
```

<div class="admonition note">
<p class="admonition-title"></p>
   A quel nombre en base 10 correpond le nombre en base 2 : 00101101 ? Comment écrit-on 24 en base 2 ?
</div>

### Sur votre machine

Nous allons créer plusieurs variables et étudier comment elles sont stockées en mémoire. On travaille avec **PyTorch** (les `tensor`), la librairie qu'on utilisera tout au long du cours.

In [14]:
import torch

L'epsilon d'une machine correspond au maximum de l'erreur d'arrondi qui peut être faite sur un calcul. Les fractions peuvent également ne pas être stockées exactement à cause de leur encodage en fractions binaires (base 2). En PyTorch, `torch.finfo(torch.float32).eps` donne cet epsilon pour les flottants 32 bits.

<div class="admonition note">
<p class="admonition-title"></p>
   Executez les instructions suivantes et étudiez les sorties. Dans ces 3 exemples, la fraction 1/10 est elle exactement représentée en mémoire de l'ordinateur ?
   
```

   1.2 - 1.1

   0.1 + 0.1 + 0.1 == 0.3
   
   format(0.1, '.17f')
   
   ```
</div>


In [16]:
1.2 - 1.1
0.1 + 0.1 + 0.1 == 0.3
format(0.1, '.17f')

'0.10000000000000001'

On s'intéresse maintenant à la place nécessaire pour stocker des variables. **PyTorch** propose plusieurs types de tenseurs. Le type ``uint8`` stocke un entier sur 8 bits ; l'instruction ``torch.iinfo(torch.uint8)`` renvoie les valeurs minimale et maximale admissibles pour ce type.

<div class="admonition note">
<p class="admonition-title"></p>

Quelles sont les valeurs minimale et maximale possibles pour le type ``torch.uint8`` ?

</div>

In [15]:
torch.iinfo(torch.uint8)

iinfo(min=0, max=255, dtype=uint8)

<div class="admonition note">
<p class="admonition-title"></p>

Expliquer les résultats de l'instruction suivante :

</div>

In [9]:
a = torch.tensor(255, dtype=torch.uint8)
print("a =", a.item(), " et  a + 1 =", (a + 1).item())
print("octets pour stocker la variable :", a.element_size() * a.nelement())

a = 255  et  a + 1 = 0
octets pour stocker la variable : 1


Cela s'explique par le fait que 255 a pour valeur 11111111 en binaire, ajouter 1 remet tout l'octet à 00000000, soit 0

**Combien pèse un modèle ?** Un modèle de deep learning, c'est surtout des **paramètres** (des flottants). En `float32`, chaque paramètre pèse 4 octets — d'où la règle « un modèle de *N* paramètres ≈ 4N octets ».

<div class="admonition note">
<p class="admonition-title"></p>

Un petit perceptron multi-couches a **1,2 million** de paramètres. Combien de **Mo** occupe-t-il en ``float32`` ? Et en ``float16`` ? Vérifiez votre calcul avec un tenseur ``torch.zeros(n, dtype=…)`` et ``element_size()``.

</div>

En float32, un tel model devrait peser 4*1.2 = 4.8 milions d'octet soit 4.8 Mo
En float16, qui est donc deux fois plus petit, un tel model devrait peser 2.4 Mo

In [20]:
torch.zeros(1200000, dtype=torch.float16).element_size()

2

### Le tenseur, l'objet central du cours

On a manipulé des tenseurs pour parler de mémoire. Regardons de plus près cet objet : c'est **la** structure de données qu'on utilisera dans **tous les TD suivants** — pour représenter les données, les paramètres d'un modèle et leurs gradients. Un tenseur est un tableau multidimensionnel caractérisé par trois attributs : sa forme (``shape``), son type (``dtype``) et l'appareil où il vit (``device``).

In [23]:
x = torch.arange(12).reshape(3, 4)
print(x)
print("forme :", x.shape, "| type :", x.dtype, "| device :", x.device)
print("nb dimensions :", x.ndim, "| nb éléments :", x.numel())

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
forme : torch.Size([3, 4]) | type : torch.int64 | device : cpu
nb dimensions : 2 | nb éléments : 12


<div class="admonition note">
<p class="admonition-title"></p>

Prédisez la sortie **avant** d'exécuter, puis vérifiez : que renvoient ``x.reshape(2, 6)``, ``x.view(-1)`` (aplatir en 1D), ``x[0]`` (1re ligne) et ``x[:, 1]`` (2e colonne) ?

</div>

x.reshape(2, 6) va renvoyer tensor([[ 0,  1,  2,  3,  4,  5],
                                    [ 6,  7,  8,  9, 10, 11]])
x.view(-1) va renvoyer tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])
x[0] va renvoyer tensor([0, 1, 2, 3])
x[:, 1] va renvoyer tensor([1, 5, 9])

In [27]:
x.reshape(2, 6)
x.view(-1)
x[0]
x[:, 1]

tensor([1, 5, 9])

**Broadcasting.** Pour les opérations élément par élément, PyTorch aligne automatiquement des formes différentes — indispensable, par exemple, pour ajouter un même biais à toutes les lignes d'un lot d'exemples.

In [28]:
X = torch.ones(3, 4)                    # 3 exemples × 4 caractéristiques
b = torch.tensor([10., 20., 30., 40.])  # un biais par caractéristique
print(X + b)   # b est « diffusé » (broadcast) sur les 3 lignes

tensor([[11., 21., 31., 41.],
        [11., 21., 31., 41.],
        [11., 21., 31., 41.]])


<div class="admonition note">
<p class="admonition-title"></p>

Pourquoi ``X + b`` fonctionne-t-il alors que les formes diffèrent ? Testez ensuite ``b = torch.tensor([1., 2., 3.])`` : que se passe-t-il, et pourquoi ?

</div>

In [ ]:
X = torch.ones(3, 4)
b = torch.tensor([1., 2., 3.])
print(X + b)  

RuntimeError: The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 1

python essaie d'ajouter b à X en tant que nouvelle ligne, et bugue parceque b est de taille 3 alors que x a 4 colonnes

### Reproductibilité : fixer le hasard

Beaucoup d'étapes du deep learning utilisent du **hasard** : initialisation des poids d'un modèle, mélange des données, *dropout*… Pour obtenir des résultats **comparables** d'une exécution à l'autre (et débugger sereinement), on fixe la *graine* (``seed``) du générateur aléatoire. En TD4 et TD5, vous verrez que l'initialisation peut tout changer : on prendra donc l'habitude de fixer la seed en début de notebook.

In [ ]:
torch.manual_seed(0)
print(torch.randn(3))
torch.manual_seed(0)
print(torch.randn(3))   # exactement les mêmes valeurs

tensor([ 1.5410, -0.2934, -2.1788])
tensor([ 1.5410, -0.2934, -2.1788])


<div class="admonition note">
<p class="admonition-title"></p>

Relancez la cellule plusieurs fois : les valeurs bougent-elles ? Retirez ensuite les deux ``torch.manual_seed(0)`` et relancez : que constatez-vous ?

</div>

In [33]:
print(torch.randn(3))
print(torch.randn(3))

tensor([ 0.5684, -1.0845, -1.3986])
tensor([ 0.4033,  0.8380, -0.7193])


Les valeurs obtenues sont bien différentes 

## Mémoire d'un ordinateur

### Sur votre machine

La librarie ``psutil`` permet de récupérer des informations concernant l'utilisation de la mémoire de l'ordinateur. 

<div class="admonition note">
<p class="admonition-title"></p>

Quelle est la RAM disponible sur votre ordinateur ? Traduire la mémoire totale indiquée par la sortie de la cellule en GiO et vérifiez que cela correpond à la mémoire "officielle" indiquée sur votre machine.
```python
import psutil
psutil.virtual_memory()
```
</div>

In [34]:
import psutil
psutil.virtual_memory()

svmem(total=24887451648, available=14428778496, percent=42.0, used=10458673152, free=5907451904, active=9340600320, inactive=6248845312, buffers=785932288, cached=10903830528, shared=1127030784, slab=1319665664)

mon pc a 24 presque 25 Go de RAM, comme indiqué par le constructeur

<div class="admonition note">
<p class="admonition-title"></p>

L'instruction ``torch.arange(m)`` génère une suite de nombres de 0 à m-1.
Combien d'octets occupe le tenseur `x` suivant ?

```python
x = torch.arange(1000*1000, dtype=torch.int32)
print("octets :", x.element_size() * x.nelement())
```
</div>

In [35]:
x = torch.arange(1000*1000, dtype=torch.int32)
print("octets :", x.element_size() * x.nelement())

octets : 4000000


<div class="admonition note">
<p class="admonition-title"></p>
Et lorsque le tenseur `x` est stocké en ``torch.int64`` au lieu de ``torch.int32`` ? Expliquez pourquoi.
</div>

In [36]:
x = torch.arange(1000*1000, dtype=torch.int64)
print("octets :", x.element_size() * x.nelement())

octets : 8000000


<div class="admonition note">
<p class="admonition-title"></p>
Créez un tenseur nommé ``x`` qui occupe 2 fois moins que votre RAM disponible (pour moi, la RAM est de 32 Gio, soit 32 * 1024 * 1024 * 1024 octets). Par exemple ``x = torch.zeros(taille, dtype=torch.float32)``. Vérifiez que la mémoire occupée correspond bien (Gestionnaire des tâches sous Windows, Moniteur d'activité sous macOS, ou ``top -o MEM`` sous Linux).
</div>

In [37]:
x = torch.zeros(3*1024*1024*1024, dtype=torch.float32)


<div class="admonition note">
<p class="admonition-title"></p>

Executez maintenant l'instruction 

```python
del x
```
et étudiez la quantité de RAM utilisée par Python. Que constatez vous ?
</div>


In [38]:
del x

<div class="admonition note">
<p class="admonition-title"></p>

Que va-t-il se passer si on crée un tenseur 2 fois plus grand que la mémoire disponible ? Créez un tel tenseur et regardez ce qui se passe dans votre moniteur d'activité. N'oubliez pas de le supprimer une fois que c'est fait.

</div>

In [39]:
x = torch.zeros(12*1024*1024*1024, dtype=torch.float32)

RuntimeError: [enforce fail at alloc_cpu.cpp:127] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 51539607552 bytes. Error code 12 (Cannot allocate memory)

<div class="admonition note">
<p class="admonition-title"></p>
Quel est le nombre de CPUs disponibles sur votre machine ? Vous pouvez utiliser l'instruction suivante pour avoir la réponse :

```python
import os
os.cpu_count()
```
</div>

In [40]:
import os
os.cpu_count()

12

### Un vrai jeu de données : BreizhCrops

Le **fil rouge** des TD suivants est **BreizhCrops** : des séries temporelles satellites (Sentinel-2) servant à classer des parcelles agricoles en Bretagne — exactement le genre de données massives de ce cours. Regardons son **empreinte mémoire**, la question au cœur de cette section.

In [ ]:
# 1re fois seulement (téléchargement) : décommentez la ligne suivante
!sudo pip install --quiet breizhcrops geopandas
from breizhcrops import BreizhCrops

dataset = BreizhCrops(region="belle-ile", preload_ram=True)
x, y, field_id = dataset[0]        # une parcelle : série temporelle + label
print("nombre de parcelles :", len(dataset))
print("une série temporelle :", x.shape, x.dtype)   # (dates × bandes spectrales)
print("mémoire d'une série  :", x.element_size() * x.nelement(), "octets")

[sudo] Mot de passe de charles : 

<div class="admonition note">
<p class="admonition-title"></p>

Combien y a-t-il de parcelles ? Quelle est la forme d'une série temporelle (nombre de dates × nombre de bandes spectrales) ? Estimez la **mémoire totale** du jeu chargé en RAM (``preload_ram=True``) et comparez-la à la RAM que vous avez mesurée au début de cette section : tient-il confortablement en mémoire ?

</div>

### Avec Google Colab

On va maintenant travailler avec [Google Colab](https://colab.research.google.com/), qui permet d'écrire et d'exécuter du code Python dans un navigateur, avec un accès au GPU.

<div class="admonition note">
<p class="admonition-title"></p>

Par défaut, quelle est le nombre de cpu alloués par notebook ? Est ce différent de votre machine ?
</div>

<div class="admonition note">
<p class="admonition-title"></p>
Exécutez les instructions suivantes et expliquez le résultat.

```python
x = torch.arange(1024*1024*1024, dtype=torch.float32)
```

et
```python
del x
x = torch.arange(1024*1024*1024*16, dtype=torch.float32)
```

</div>

Certaines librairies permettent de faire du calcul sur GPU — c'est le cas de **PyTorch**.

<div class="admonition note">
<p class="admonition-title"></p>

Exécutez les commandes suivantes :

```python
import torch

print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))
```
</div>

On peut demander à accéder à un GPU (en accès limité !) en allant dans le menu `Execution > Modifier le type d'exécution > Accélérateur matériel T4 GPU`. 
<div class="admonition note">
<p class="admonition-title"></p>
Effectuez la manipulation et relancez le code précédent. Combien de GPUs sont-ils disponibles ?
</div>

Un GPU accélère certains calculs en les répartissant sur ses (très) nombreux cœurs — en particulier les **multiplications de matrices**, l'opération de base d'un réseau de neurones. Le code suivant multiplie deux matrices 5000×5000 sur CPU puis sur GPU (`%timeit` donne un temps moyen sur plusieurs répétitions).

<div class="admonition note">
<p class="admonition-title"></p>

Lancez le code suivant et expliquez la différence de temps :

```python
a = torch.randn(5000, 5000)
b = torch.randn(5000, 5000)

# sur CPU
%timeit -n 5 -r 3 torch.mm(a, b)

# sur GPU (si disponible)
if torch.cuda.is_available():
    a_gpu, b_gpu = a.to("cuda"), b.to("cuda")
    %timeit -n 5 -r 3 (torch.mm(a_gpu, b_gpu), torch.cuda.synchronize())
```
</div>

**Un GPU sans Colab ?** Sur votre machine aussi, peut-être : les Mac à puce Apple Silicon exposent un GPU via le backend ``mps``. Le **motif standard**, qu'on réutilisera dans tous les TD, choisit automatiquement le meilleur appareil disponible.

<div class="admonition note">
<p class="admonition-title"></p>

Exécutez ce motif, puis provoquez le **piège classique** (des tenseurs sur des appareils différents) :

```python
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("appareil utilisé :", device)

a = torch.randn(3)                 # sur le CPU
b = torch.randn(3, device=device)  # sur le GPU si disponible
a + b   # RuntimeError si device n'est pas "cpu" : tout doit être au même endroit
```
</div>

**Pourquoi la vectorisation change tout.** Comparez une boucle Python à l'opération vectorisée équivalente (même calcul, un produit scalaire) :

<div class="admonition note">
<p class="admonition-title"></p>

Combien de fois plus rapide est la version vectorisée ? Pourquoi ?

```python
n = 1_000_000
a, b = torch.randn(n), torch.randn(n)

%timeit -n 1 -r 1 sum(a[i] * b[i] for i in range(n))   # boucle Python interprétée
%timeit -n 100 -r 3 torch.dot(a, b)                    # vectorisé (code compilé)
```
</div>

**Le GPU n'aide que si le calcul est assez gros.** Le gain grandit avec la taille des matrices — sur les petites, le coût de transfert et de lancement du calcul domine.

<div class="admonition note">
<p class="admonition-title"></p>

Mesurez le temps d'une multiplication de matrices pour des tailles croissantes (sur CPU ci-dessous ; refaites-le sur GPU dans Colab et comparez). Sur les petites matrices, le GPU peut être **plus lent** — pourquoi ?

```python
import time
for N in [64, 512, 2048]:
    a, b = torch.randn(N, N), torch.randn(N, N)
    t = time.time(); torch.mm(a, b); cpu = time.time() - t
    print(f'N={N:5d}  CPU={cpu*1e3:7.1f} ms')
```
</div>

<div class="admonition attention">
Ce qu'on a vu jusqu'à présent : 

- la RAM permet de stocker des données

- le processeur (CPU ou GPU) permet de faire des calculs

- les CPU et GPU sont constitués de plus de coeurs, le GPU peut permettre d'accélerer les calculs
</div>


## Threads

On va maintenant travailler en asynchrone, c'est-à-dire lorsque chaque ordinateur/noeud/coeur fonctionne indépendamment, sans attendre les résultats des autres, et renvoie le résultat une fois qu'il est terminé. La librairie ``concurrent.futures`` de python permet de faire du calcul asynchrone, soit avec des threads, soit avec des process.  On va voir ici le fonctionnement des threads. 

<div class="admonition note">
<p class="admonition-title"></p>

Importer la librairie.

</div>

On va maintenant considérer la fonction ``wait_and_add`` qui permet d'attendre 2 secondes (par défaut) avant d'ajouter 1 à un chiffre donné en entrée. 

In [22]:
from time import sleep

def wait_and_add(a, delay=2):
    sleep(delay)
    return a + 1

wait_and_add(1,1)

2

On souhaite ajouter +1 à la liste de nombres suivante :
```python
l = list(range(10))
```
<div class="admonition note">
<p class="admonition-title"></p>
Écrivez une boucle qui le fait et, avec ``%%time`` (placée en début de cellule, elle retourne le temps d'exécution), calculez le temps nécessaire pour traiter toute la liste (en essayant de deviner le résultat au préalable !).
</div>

On va maintenant essayer de tirer parti des différents coeurs de notre CPU pour lancer le calcul en asynchrone. Pour cela, on va utiliser  ``ThreadPoolExecutor``, comme dans le code suivant :

In [12]:
from concurrent.futures import ThreadPoolExecutor

In [ ]:
%%time
with ThreadPoolExecutor(max_workers=1) as executor:
    executor.map(wait_and_add, l)

L'option ``max_workers`` permet de découper l'exécution en un certain nombre de threads sur lesquels faire les calculs de façon asynchrone. Vous pouvez également regarder la [doc associée](https://docs.python.org/3/library/concurrent.futures.html#threadpoolexecutor) pour mieux comprendre l'instruction. 

<div class="admonition note">
<p class="admonition-title"></p>
A votre avis, quel va être le temps d'exécution si on choisit ``max_workers=2`` ?
</div>

<div class="admonition note">
<p class="admonition-title"></p>

Et si on choisit ``max_workers=10`` ? et plus que 10 ?
</div>

<div class="admonition tip">
<p class="admonition-title"></p>

**→ pour le deep learning :** on retrouvera exactement cette idée. Le ``DataLoader`` de PyTorch utilise ``num_workers`` pour charger et préparer les données **en parallèle** (plusieurs process) pendant que le GPU calcule — c'est le même principe que le ``max_workers`` vu ici.

</div>